# Milestone 6 — Local LLM RAG

Pipeline: **query → Hybrid Retrieval M5 → evidence prompt → Qwen 4-bit → answer**. Notebook chỉ load index có sẵn, không rebuild BM25, embedding hoặc FAISS. M6 chưa triển khai citation chuẩn hóa và legal comparison.

## 1. Mount Google Drive và setup Colab T4

In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path('/content/drive/MyDrive/Master/IT2043/Project/Final_Project/vn-landlaw-qa')
assert PROJECT_ROOT.exists(), f'Không tìm thấy repo: {PROJECT_ROOT}'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Project root:', PROJECT_ROOT)

Project root: /content/drive/MyDrive/Master/IT2043/Project/Final_Project/vn-landlaw-qa


In [3]:
# !python -m pip install -q -r requirements/retrieval.txt

In [4]:
# %pip install -q faiss-cpu sentence-transformers transformers accelerate bitsandbytes

In [5]:
%pip install -q \
  "transformers==4.48.3" \
  "sentence-transformers==3.4.1" \
  "accelerate==1.3.0" \
  "bitsandbytes==0.45.2" \
  "faiss-cpu==1.10.0"

In [6]:
%pip uninstall -y torchvision

In [7]:
# %pip install -q -U -r requirements/generation.txt

In [16]:
import torch
assert torch.cuda.is_available(), 'Hãy chọn Runtime > Change runtime type > T4 GPU'
gpu = torch.cuda.get_device_properties(0)
print(f'GPU: {gpu.name} | VRAM: {gpu.total_memory / 1024**3:.2f} GB')

GPU: Tesla T4 | VRAM: 14.56 GB


## 2. Load Hybrid Retriever M5 — không rebuild

In [29]:
CHUNKS_PATH = PROJECT_ROOT / 'data/processed/chunks.json'
BM25_INDEX_PATH = PROJECT_ROOT / 'indexes/bm25/bm25_index.json.gz'
FAISS_DIR = PROJECT_ROOT / 'indexes/faiss'
REQUIRED = [
    CHUNKS_PATH, BM25_INDEX_PATH, FAISS_DIR / 'faiss.index',
    FAISS_DIR / 'vector_metadata.json', FAISS_DIR / 'embedding_manifest.json',
]
missing = [str(path) for path in REQUIRED if not path.exists()]
assert not missing, 'Thiếu artifacts M3–M5:\n' + '\n'.join(missing)
print('Đã tìm thấy toàn bộ artifacts; chỉ thực hiện load.')

Đã tìm thấy toàn bộ artifacts; chỉ thực hiện load.


In [10]:
modules = ["faiss", "numpy", "torch", "sentence_transformers"]

for name in modules:
    try:
        module = __import__(name)
        print("OK:", name, getattr(module, "__version__", ""))
    except Exception as exc:
        print("ERROR:", name, repr(exc))

OK: faiss 1.10.0
OK: numpy 2.5.3
OK: torch 2.14.1+cu130


: 

: 

In [18]:
from landlaw_rag.retrieval import (
    BM25Index, VectorIndex, BM25Retriever, VectorRetriever,
    HybridConfig, HybridRetriever,
)

bm25_index = BM25Index.load(BM25_INDEX_PATH)
vector_index = VectorIndex.load(
    FAISS_DIR, device='cuda', load_encoder=True, chunks_path=CHUNKS_PATH
)
hybrid_retriever = HybridRetriever(
    BM25Retriever(bm25_index),
    VectorRetriever(vector_index),
    HybridConfig(bm25_top_n=20, vector_top_n=20, final_top_k=6, rrf_k=60),
)

def hybrid_search(query: str, top_k: int = 6) -> list[dict]:
    return hybrid_retriever.search(query, top_k=top_k)

print(f'BM25: {len(bm25_index.documents):,} chunks | FAISS: {vector_index.index.ntotal:,} vectors')

BM25: 921 chunks | FAISS: 921 vectors


## 3. Load Qwen 7B bằng NF4 4-bit

Checkpoint `Qwen/Qwen2.5-7B-Instruct` là bản instruction multilingual có hỗ trợ tiếng Việt. NF4 4-bit với compute FP16 phù hợp GPU T4 16 GB.

In [19]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
MAX_CONTEXT_TOKENS = 5500
MAX_INPUT_TOKENS = 7000
MAX_NEW_TOKENS = 512
TOP_K = 6

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map='auto',
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
model.eval()
model.generation_config.do_sample = False
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None
print(f'Model: {MODEL_NAME}')
print(f'Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')
print(f'Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB')
print(f'Peak:      {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB')

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model: Qwen/Qwen2.5-7B-Instruct
Allocated: 9.45 GB
Reserved:  11.55 GB
Peak:      11.60 GB


## 4. Context, prompt và hàm `rag_answer(query)`

In [30]:
import importlib
import landlaw_rag.generation.rag as rag_module
import landlaw_rag.generation as generation_module

importlib.reload(rag_module)
importlib.reload(generation_module)

from landlaw_rag.generation import (
    build_context,
    build_messages,
    detect_version_intent,
    expand_article_context,
    filter_results_by_version,
    format_source,
)

print("Reload thành công")

Reload thành công


In [31]:
from landlaw_rag.generation import (
    build_context, build_messages, detect_version_intent, expand_article_context,
    filter_results_by_version, format_source,
)

import json

all_chunks = json.loads(CHUNKS_PATH.read_text(encoding='utf-8'))

def _print_context_stage(label: str, items: list[dict]) -> None:
    print(f'\n{label} ({len(items)} chunks)')
    for item in items:
        marker = ' [EXPANDED]' if item.get('expanded') else ''
        print(f"- {format_source(item.get('metadata', {}))}{marker}")

def retrieve_context(
    query: str, top_k: int = TOP_K, debug: bool = False
) -> tuple[str, list[dict]]:
    # Lấy dư candidates rồi lọc theo phiên bản trước khi chọn Top-k cuối.
    candidates = hybrid_search(query, top_k=max(top_k * 4, 20))
    results = filter_results_by_version(candidates, query)[:top_k]
    expanded = expand_article_context(
        results, all_chunks, MAX_CONTEXT_TOKENS, tokenizer=tokenizer
    )
    if debug:
        _print_context_stage('TRƯỚC EXPANSION', results)
        _print_context_stage('SAU EXPANSION', expanded)
    return build_context(expanded, tokenizer, max_context_tokens=MAX_CONTEXT_TOKENS)

In [32]:
import time

@torch.inference_mode()
def rag_answer(
    query: str, top_k: int = TOP_K, max_new_tokens: int = MAX_NEW_TOKENS,
    debug_context: bool = False,
) -> dict:
    query = query.strip()
    if not query:
        raise ValueError('Query không được rỗng')
    total_start = time.perf_counter()

    retrieval_start = time.perf_counter()
    context, sources = retrieve_context(query, top_k=top_k, debug=debug_context)
    torch.cuda.synchronize()
    retrieval_seconds = time.perf_counter() - retrieval_start

    messages = build_messages(query, context)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(
        prompt, return_tensors='pt', truncation=True, max_length=MAX_INPUT_TOKENS
    ).to(model.device)

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    generation_start = time.perf_counter()
    generated = model.generate(
        **inputs,
        max_new_tokens=min(max_new_tokens, MAX_NEW_TOKENS),
        do_sample=False,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id,
    )
    torch.cuda.synchronize()
    generation_seconds = time.perf_counter() - generation_start
    output_tokens = generated[0, inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(output_tokens, skip_special_tokens=True).strip()
    total_seconds = time.perf_counter() - total_start

    return {
        'query': query,
        'answer': answer,
        'sources': sources,
        'input_tokens': int(inputs['input_ids'].shape[1]),
        'output_tokens': int(output_tokens.shape[0]),
        'retrieval_seconds': retrieval_seconds,
        'generation_seconds': generation_seconds,
        'total_seconds': total_seconds,
        'peak_generation_vram_gb': torch.cuda.max_memory_allocated() / 1024**3,
    }

In [33]:
def print_rag_result(result: dict) -> None:
    print('CÂU HỎI:', result['query'])
    print('\nTRẢ LỜI:\n', result['answer'])
    print(f"\nTokens: input={result['input_tokens']} | output={result['output_tokens']}")
    print(
        f"Timing: retrieval={result['retrieval_seconds']:.3f}s | "
        f"generation={result['generation_seconds']:.3f}s | total={result['total_seconds']:.3f}s"
    )
    print(f"Peak generation VRAM: {result['peak_generation_vram_gb']:.2f} GB")
    print('Evidence chunks:', len(result['sources']))
    for source in result['sources']:
        label = format_source(source.get('metadata', {}))
        if source.get('expanded'):
            print(f"  [EXPANDED] {label}")
        else:
            score = source.get('final_score')
            score_text = f'{score:.6f}' if score is not None else 'n/a'
            print(f"  #{source.get('rank')} {label} | final_score={score_text}")

## 5. Thử nghiệm QA dựa trên evidence

In [34]:
TEST_QUESTIONS = [
    # Hiện hành: evidence Luật 2013 phải bị loại.
    'Điều kiện để cá nhân được chuyển nhượng quyền sử dụng đất là gì?',
    # Lịch sử: chỉ giữ evidence Luật 45/2013/QH13.
    'Theo Luật Đất đai 2013, điều kiện chuyển nhượng quyền sử dụng đất là gì?',
]

for question in TEST_QUESTIONS:
    result = rag_answer(question, debug_context=True)
    print('VERSION MODE:', detect_version_intent(question))
    print_rag_result(result)
    print('\n' + '=' * 100 + '\n')


TRƯỚC EXPANSION (6 chunks)
- Luật Đất đai số 31/2024/QH15 → Chương III - QUYỀN VÀ NGHĨA VỤ CỦA NGƯỜI SỬ DỤNG ĐẤT → Mục 5 - ĐIỀU KIỆN THỰC HIỆN CÁC QUYỀN → Điều 45. Điều kiện thực hiện các quyền chuyển đổi, chuyển nhượng, cho thuê, cho thuê lại, thừa kế, tặng cho quyền sử dụng đất; thế chấp, góp vốn bằng quyền sử dụng đất; nhận chuyển nhượng, nhận tặng cho quyền sử dụng đất → Khoản 7
- Luật Đất đai số 31/2024/QH15 → Chương III - QUYỀN VÀ NGHĨA VỤ CỦA NGƯỜI SỬ DỤNG ĐẤT → Mục 5 - ĐIỀU KIỆN THỰC HIỆN CÁC QUYỀN → Điều 45. Điều kiện thực hiện các quyền chuyển đổi, chuyển nhượng, cho thuê, cho thuê lại, thừa kế, tặng cho quyền sử dụng đất; thế chấp, góp vốn bằng quyền sử dụng đất; nhận chuyển nhượng, nhận tặng cho quyền sử dụng đất → Khoản 3
- Luật Đất đai số 31/2024/QH15 → Chương III - QUYỀN VÀ NGHĨA VỤ CỦA NGƯỜI SỬ DỤNG ĐẤT → Mục 5 - ĐIỀU KIỆN THỰC HIỆN CÁC QUYỀN → Điều 45. Điều kiện thực hiện các quyền chuyển đổi, chuyển nhượng, cho thuê, cho thuê lại, thừa kế, tặng cho quyền sử dụng đất;

## Phạm vi M6

Notebook này tạo câu trả lời grounded theo evidence và từ chối bằng câu `Chưa đủ căn cứ trong dữ liệu truy xuất.` khi context thiếu. Evidence được cô lập theo ba chế độ: hiện hành, Luật 2013 và so sánh 2013–hiện hành. Các nhãn `[Nguồn n]` giúp model phân biệt context nội bộ; M6 chưa chuẩn hóa citation pháp lý.